# DanmakuRender：Colab 单场录制与多 P 投稿

本地触发或手动启动，录制、ASS 渲染、分段备份、可选 B 站多 P 投稿及全场合并归档。公开配置默认关闭投稿，只有私有配置明确启用才会自动投稿。

按顺序运行。GPU、Drive 授权、观看 Cookie、投稿 Cookie 和账号配置由本人准备。Cookie 不写入 Notebook。运行完毕保留文件和运行时。


In [ ]:
SESSION = None
SETUP_PHASE = "invalid"
from pathlib import Path
import subprocess
import sys

REPOSITORY = "https://github.com/zaixiakongyiji/DanmakuRenderForColab.git"
BRANCH = "codex/colab-poc"
PROJECT = Path("/content/DanmakuRenderForColab")

def checked(command, **kwargs):
    return subprocess.run(command, check=True, **kwargs)

if not PROJECT.exists():
    checked(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(PROJECT)])
else:
    actual_remote = subprocess.check_output(["git", "-C", str(PROJECT), "remote", "get-url", "origin"], text=True).strip()
    actual_branch = subprocess.check_output(["git", "-C", str(PROJECT), "branch", "--show-current"], text=True).strip()
    if actual_remote != REPOSITORY or actual_branch != BRANCH:
        raise RuntimeError("已有目录不对应选定仓库或分支，请先检查目录。")
    print("使用现有检出；不覆盖运行中的文件。更新代码前请结束当前任务。")
COMMIT = subprocess.check_output(["git", "-C", str(PROJECT), "rev-parse", "HEAD"], text=True).strip()
print("本次源码 commit:", COMMIT)

SETUP_PHASE = "source"


## 安装依赖

首版只安装 B 站录制及渲染路径所需依赖。系统 FFmpeg 必须实际通过 NVENC + ASS 预检；失败时检查驱动和 FFmpeg 版本，不自动改成 CPU 编码。


In [ ]:
SESSION = None
previous_phase = globals().get('SETUP_PHASE')
SETUP_PHASE = 'invalid'
if previous_phase != 'source':
    raise RuntimeError('请先成功运行源码准备单元格')
checked(["apt-get", "update", "-qq"])
checked(["apt-get", "install", "-y", "ffmpeg", "fontconfig", "fonts-noto-cjk"])
checked([sys.executable, "-m", "pip", "install", "-r", str(PROJECT / "colab_support/requirements.txt")])
checked(["fc-cache", "-f"])
SETUP_PHASE = 'dependencies'


## 授权 Drive

授权后，本 Notebook 运行的代码可访问挂载的 Drive。观看 Cookie 只用于读取 B 站直播流，备份只包含录像、ASS、成品、运行清单及白名单事件日志。


In [ ]:
SESSION = None
previous_phase = globals().get('SETUP_PHASE')
SETUP_PHASE = 'invalid'
if previous_phase != 'dependencies':
    raise RuntimeError('请先成功运行依赖安装单元格')
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
COOKIE_FILE = DRIVE_ROOT / "DMRColab/credentials/bilibili.json"
if not COOKIE_FILE.is_file():
    raise FileNotFoundError("请手动上传现有登录 JSON 到 MyDrive/DMRColab/credentials/bilibili.json，再重新运行此单元格。")
print("已找到观看 Cookie 文件，内容不会显示。")
SETUP_PHASE = 'drive'


## 运行参数

默认每 3600 秒一个录制分段和一个投稿 P。短试跑可改为 300 秒，MAX_RECORD_HOURS 改为 0.2。不足 120 秒的分段仍备份但不投稿。

自动触发读取私有 Drive 的 active.json 及对应运行请求；手动运行填写 ROOM_URL，每次参数设置生成新编号。私有 upload.yml 和投稿 Cookie 可在预检前准备，公开仓库的示例默认禁用。


In [ ]:
SESSION = None
if globals().get('SETUP_PHASE') != 'drive':
    raise RuntimeError('请先按顺序完成源码、依赖和 Drive 单元格')
import sys
sys.path.insert(0, str(PROJECT))
from colab_support.notebook_session import prepare_session

ROOM_URL = ""  # 手动运行填写；留空则读取本地触发请求
SEGMENT_SECONDS = 3600
INITIAL_WAIT_MINUTES = 15
OFFLINE_GRACE_MINUTES = 3
MAX_RECORD_HOURS = 12
DRAIN_TIMEOUT_HOURS = 2
MIN_FREE_GIB = 10

SESSION = prepare_session(PROJECT, DRIVE_ROOT, COOKIE_FILE, ROOM_URL, sys.executable,
    segment=SEGMENT_SECONDS, initial_wait=INITIAL_WAIT_MINUTES * 60,
    offline_grace=OFFLINE_GRACE_MINUTES * 60, max_record=MAX_RECORD_HOURS * 3600,
    drain_timeout=DRAIN_TIMEOUT_HOURS * 3600, min_free_gib=MIN_FREE_GIB)
RUN_ID, RUN_DIR, BACKUP_DIR = SESSION.run_id, SESSION.run_dir, SESSION.backup_dir
print('运行编号:', RUN_ID)
print('运行目录:', RUN_DIR)
print('Drive 备份目录:', BACKUP_DIR)


In [ ]:
if SESSION is None:
    raise RuntimeError('参数单元格未成功，禁止沿用旧变量')
SESSION.preflight(checked)
from IPython.display import Video, display
display(Video(str(SESSION.run_dir / 'preflight/rendered.mp4'), embed=True))


## 开始运行

下面的单元格持续等待任务完成，并显示状态、分段数和失败项。首次未开播最多等 15 分钟；下播持续 3 分钟后等待尾段、渲染、自动合并和备份。

**需要提前停止时，点一次停止单元格**：捕获中断后请求受控停止，仍等待已有分段处理完。第一次中断只创建 `STOP` 并等待收尾；再次中断明确创建 `FORCE_STOP`，才请求强制结束，不算正常完成。遇到浏览器未能传递中断时，可以在运行时终端执行 `touch /content/dmr-runs/<运行编号>/STOP`，然后重新查看结果。

此单元格保持前台执行，不使用后台保活脚本；运行完毕后保留运行时，由你检查并手动释放。

启动阶段会显示 `room_info`、`stream_select`、`probe`、`danmaku_ready` 和 `recorder_start`。`probe` 最长可能等待约 15 秒，此时尚未开始写入录像。已运行过的目录不能直接重跑，请先重新执行参数单元格。


In [ ]:
if SESSION is None:
    raise RuntimeError('参数单元格未成功')
from colab_support.notebook import run_recording
returncode = SESSION.run(run_recording)
print('退出码:', returncode)
if returncode != 0:
    print('本次未完整成功，请查看清单；保留未备份文件和 unknown 投稿证据。')


## 检查结果并记录 CU

分别检查 recording_backup、merge 和 upload。启用投稿时，至少一个有效 P 且所有应投分段确认提交或短段跳过，才算投稿完成；submitted 不代表审核通过或已公开。最终成功还要求退出码为 0、清单与文件可读取。

全场合并文件位于 merged/complete.mp4，仅用于归档。检查音视频、弹幕同步和最后分段。CU 由界面记录，不从 GPU 型号推算。


In [ ]:
import json

manifest_path = RUN_DIR / "manifest.json"
if not manifest_path.exists():
    raise RuntimeError("尚未生成运行清单，请查看本地 console.log。")
result = json.loads(manifest_path.read_text(encoding="utf-8"))
print("运行状态:", result.get("status"), "退出原因:", result.get("stop_reason"))
print("投稿:", result.get("upload"), "投稿错误:", result.get("upload_errors", []))
print("计数:", json.dumps(result.get("counts", {}), ensure_ascii=False))
merge = result.get("merge", {})
print("合并状态:", merge.get("status"), "备份:", merge.get("backup", {}).get("status"))
if merge.get("path"):
    print("合并文件:", RUN_DIR / merge["path"],
          "时长:", round(merge.get("duration_seconds", 0), 3),
          "分段:", merge.get("segment_count"),
          "全片解码校验:", merge.get("decode_verified"))
if result.get("errors"):
    print("错误:", result["errors"])
cloud_manifest = BACKUP_DIR / "manifest.json"
print("Drive 清单可读:", cloud_manifest.is_file())
if cloud_manifest.is_file():
    cloud_result = json.loads(cloud_manifest.read_text(encoding="utf-8"))
    print("Drive 状态:", cloud_result.get("status"),
          "Drive 合并备份:", cloud_result.get("merge", {}).get("backup", {}).get("status"))
for key, segment in result.get("segments", {}).items():
    duration = segment.get("duration", 0)
    elapsed = segment.get("render_elapsed_seconds")
    print(key, "弹幕条数:", segment.get("danmaku_dialogues"),
          "渲染耗时/视频时长:", round(elapsed / duration, 3) if duration and elapsed else None)
CU_BEFORE = None  # 填入开始前界面显示的余额
CU_AFTER = None   # 填入结束后界面显示的余额
if CU_BEFORE is not None and CU_AFTER is not None:
    usage = {"cu_before": CU_BEFORE, "cu_after": CU_AFTER, "cu_used": CU_BEFORE - CU_AFTER}
    (RUN_DIR / "usage.json").write_text(json.dumps(usage, indent=2), encoding="utf-8")
    if cloud_manifest.is_file():
        sys.path.insert(0, str(PROJECT))
        from colab_support.core import verified_copy
        verified_copy(RUN_DIR / "usage.json", BACKUP_DIR / "usage.json")

检查完后，手动选择 **运行时 → 断开连接并删除运行时**。先确认需要的录像和诊断文件已经保存；临时磁盘内容不会随 Notebook 保留。
